# Unitree-Go2-Flat-MethodA-Electric — 계산 흐름 노트

본 노트의 목적은 한 정책 주기 안에서 q, q̇, τ, I 가 어떻게 전파되는지를
연구실 구성원이 **값을 직접 넣어가며** 따라가는 데 있다.

각 단계는 다음 3박자 셀로 구성된다.
1. **(A) 소스 발췌** — 프로젝트 코드를 그대로 인용 (`# 출처: <파일>:<라인>`)
2. **(B) 값 대입** — 동일한 식을 1∼2 자유도 축소 예제로 옮겨 직접 계산
3. **(C) 중간 결과** — 다음 절로 넘어가는 산출물 한 줄 요약

추측은 쓰지 않는다.


## 0. 개요

- 대상 시스템: Unitree Go2 (12 관절 사족보행 로봇), 평지 보행 task
- task ID: `Unitree-Go2-Flat-MethodA-Electric`
- 시간 척도 (코드 확인 결과):
    - 정책 주기:    **20 ms**
    - PD 재계산 주기: **5 ms**
    - 물리 적분 주기: **0.1 ms**
- "MethodA" 의 의미 (코드 확인): BE 일관 — 적분기 / Schur / Force RHS 모두
  $\beta = 1/(1 + h/\tau)$. patched mjwarp 의 `dynprm[4] = 0`.
  - 기호: $\beta$ 는 1-step 필터/implicit 적분 계수, $h$ 는 물리 적분 시간 간격,
    $\tau$ 는 필터 시정수다.
- "Electric" 의 의미 (코드 확인): 모터 전류 $I$ 를 MuJoCo 의 activation
  state (`d->act`) 에 통합한 BLDC 모터 모델 (`dyntype = filterexact` +
  Schur cross-Jacobian).

용어 정의:

- **ZOH (zero-order hold)**: 어떤 시점에 계산한 값을 다음 갱신 시점까지 상수로
  유지하는 sample-and-hold 방식이다. 예를 들어 $u_k$ 를 $t_k$ 에 계산하면
  $t \in [t_k, t_{k+1})$ 동안 $u(t) = u_k$ 로 쓴다.
  기호: $u_k$ 는 $k$번째 갱신 시점에서 계산된 값, $t_k$ 와 $t_{k+1}$ 은
  연속한 두 갱신 시각, $u(t)$ 는 실제 시간 $t$ 에 actuator/제어기가 참조하는 값이다.
- **`filterexact`**: MuJoCo actuator dynamics 타입 중 하나로, activation state 를
  1차 필터처럼 적분한다. 이 노트에서는 activation 을 전류 $I$ 로 해석하고,
  `d->ctrl` 로 들어온 목표 전류와 현재 전류의 차이를 `act_dot` 으로 만든 뒤
  `act = ...` 식으로 다음 전류를 계산한다.

시간 척도 출처:
`src/tasks/velocity/config/go2/env_cfgs.py:189-219` 와
`src/assets/robots/unitree_go2/go2_constants.py:305-306`
(`_COUPLED_SUBSTEPS = 200`, `_PD_RECOMPUTE = 50`).


## 1. 시간 척도 계층


```mermaid
flowchart TB
    P["정책 (20 ms, 1회)"] -- "q_des(t)=q_des,k" --> PD1["PD #1 (5 ms)"]
    P -- "q_des(t)=q_des,k" --> PD2["PD #2"]
    P -- "q_des(t)=q_des,k" --> PD3["PD #3"]
    P -- "q_des(t)=q_des,k" --> PD4["PD #4"]
    PD1 -- "τ_des, I_des: 50 substep 동안 cache 값 유지" --> PHY1["물리 0.1 ms × 50"]
    PD2 -- "τ_des, I_des" --> PHY2["물리 × 50"]
    PD3 -- "τ_des, I_des" --> PHY3["물리 × 50"]
    PD4 -- "τ_des, I_des" --> PHY4["물리 × 50"]
    PHY1 -- "q, q̇, I" --> PD2
    PHY2 -- "q, q̇, I" --> PD3
    PHY3 -- "q, q̇, I" --> PD4
    PHY4 -- "q, q̇, I" --> P
```

화살표에 적힌 것이 단계 사이에 전달되는 물리량이다.


## 2. 정책 단계 (20 ms)

### (A) 소스 발췌 — task 등록과 시간 설정

```python
# 출처: src/tasks/velocity/config/go2/__init__.py:81-93
register_mjlab_task(
  task_id="Unitree-Go2-Flat-MethodA-Electric",
  env_cfg=unitree_go2_flat_methoda_electric_env_cfg(
    use_velocity_action=_methoda_use_vel,
  ),
  play_env_cfg=unitree_go2_flat_methoda_electric_env_cfg(
    play=True, use_velocity_action=_methoda_use_vel,
  ),
  rl_cfg=unitree_go2_methoda_electric_ppo_runner_cfg(
    action_type=_METHODA_ACTION_TYPE,
  ),
  runner_cls=VelocityOnPolicyRunner,
)
```

```python
# 출처: src/tasks/velocity/config/go2/env_cfgs.py:189-219
def unitree_go2_flat_methoda_electric_env_cfg(
  play: bool = False,
  use_velocity_action: bool = False,
) -> ManagerBasedRlEnvCfg:
  cfg = unitree_go2_flat_env_cfg(play=play)
  cfg.scene.entities = {"robot": get_go2_methoda_robot_cfg()}
  cfg.sim.mujoco.timestep = 0.0001   # 0.1ms
  cfg.decimation = 200                # 0.1ms × 200 = 20ms policy dt
  ...
  return cfg
```

→ 정책 주기 = `timestep × decimation` = 0.1 ms × 200 = **20 ms**.


### (B) 값 대입 — 관측과 정책 출력 q_des

본 셀이 사용하는 ckpt:
`logs/rsl_rl/go2_methoda_electric/2026-04-27_23-25-13_act-pos_pdt20ms_phyDt0p1ms_tauDec4/model_1999.pt`
(iter 1999, agent.yaml 의 actor: `[512,256,128]` ELU MLP, init_std=1.0).

정책 단계에서 계산되는 목표 관절각을 다음 수식으로 둔다.

$$
a_k = \pi_\theta(o_k),
\qquad
q_{\mathrm{des},k} = q_{\mathrm{default}} + s_a\,a_k
$$

기호:
- $a_k \in \mathbb{R}^{12}$: $k$번째 정책 step 에서 정책이 출력하는 raw action
- $q_{\mathrm{des},k} \in \mathbb{R}^{12}$: PD 단계가 참조하는 목표 관절각
- $\pi_\theta$: 파라미터 $\theta$ 를 가진 MLP 정책 (`obs_normalizer` 거친 후 4-layer MLP)
- $o_k \in \mathbb{R}^{47}$: 정책 입력 관측 벡터
- $q_{\mathrm{default}}$: home 포즈 관절각 (hip=0, thigh=0.9, calf=−1.8, 다리 4개)
- $s_a = 0.25$: env.yaml `actions.joint_pos.scale`

#### 관측 $o_k$ 의 구성 (env.yaml `observations.actor`, 47 dims)

| 슬라이스 | 항목 | 차원 | 출처 함수 |
|----------|------|------|----------|
| `[0:3]`   | base_ang_vel       | 3 | `mjlab.envs.mdp.observations.builtin_sensor("robot/imu_ang_vel")` |
| `[3:6]`   | projected_gravity  | 3 | `mjlab.envs.mdp.observations.projected_gravity` |
| `[6:9]`   | command (twist)    | 3 | `mjlab.envs.mdp.observations.generated_commands("twist")` $=(v_x^{\mathrm{cmd}},v_y^{\mathrm{cmd}},\omega_z^{\mathrm{cmd}})$ |
| `[9:11]`  | phase (sin, cos)   | 2 | `src.tasks.velocity.mdp.observations.phase(period=0.6)` — stand 명령일 때 0 |
| `[11:23]` | joint_pos_rel      | 12 | $q - q_{\mathrm{default}}$ |
| `[23:35]` | joint_vel_rel      | 12 | $\dot q$ |
| `[35:47]` | last_action        | 12 | 직전 step 의 raw action $a_{k-1}$ |

12 관절은 xml DFS 순서로
`FL_hip, FL_thigh, FL_calf, FR_hip, FR_thigh, FR_calf, RL_hip, RL_thigh, RL_calf, RR_hip, RR_thigh, RR_calf`
이고, 본 노트북의 3 관절 축소 예제는 그중 `FR_hip, FR_thigh, FR_calf` (인덱스 3,4,5) 만 슬라이스해 사용한다.

#### 아래 코드의 흐름

1. 대표 관측 $o_k$ 구성: $v^{\mathrm{cmd}} = 0.5\,\mathrm{m/s}$ 정면 보행, 정자세, home 포즈, 직전 행동 0.
2. 위 관측을 ckpt 의 `obs_normalizer` (`mean`, `std`) 로 정규화.
3. ckpt 의 4-layer ELU MLP 가중치를 그 자리에서 통과시켜 raw action $a_k$ 를 얻는다 (`torch.load` 로 ckpt 를 직접 로드해 forward 한다).
4. $q_{\mathrm{des},k} = q_{\mathrm{default}} + 0.25\,a_k$ 로 12 차원 목표 관절각을 만든다.
5. FR 다리 3 개 관절만 슬라이스해 다음 절들의 입력으로 넘긴다.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# ── 1. ckpt 로드 ───────────────────────────────────────────────────────────
# 학습된 Method A 정책: 4-layer ELU MLP 47→512→256→128→12, scalar std,
# obs_normalizer 의 running mean/std 도 함께 저장돼 있다.
CKPT_PATH = (
    "logs/rsl_rl/go2_methoda_electric/"
    "2026-04-27_23-25-13_act-pos_pdt20ms_phyDt0p1ms_tauDec4/"
    "model_1999.pt"
)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = ckpt["actor_state_dict"]
print(f"loaded {CKPT_PATH}  (iter={ckpt['iter']})")

# obs_normalizer: (obs - mean) / std (학습 중 누적된 running stat)
obs_mean = sd["obs_normalizer._mean"][0]   # (47,)
obs_std  = sd["obs_normalizer._std"][0]    # (47,)

# 4-layer MLP, agent.yaml: hidden_dims=(512,256,128), activation=elu
mlp = nn.Sequential(
    nn.Linear(47, 512), nn.ELU(),
    nn.Linear(512, 256), nn.ELU(),
    nn.Linear(256, 128), nn.ELU(),
    nn.Linear(128, 12),
)
for src_idx, dst in [(0, 0), (2, 2), (4, 4), (6, 6)]:
    mlp[dst].weight.data = sd[f"mlp.{src_idx}.weight"]
    mlp[dst].bias.data   = sd[f"mlp.{src_idx}.bias"]
mlp.eval()

# ── 2. 관측 구성 (env.yaml observations.actor, 총 47 dims) ────────────────────
# 이 값을 바꿔보세요. 단, normalizer 의 분포 (학습 중 누적된 mean/std) 와 너무
# 멀어지면 정책 출력이 nominal 동작에서 벗어난다.
ang_vel    = np.array([0.0, 0.0, 0.0])      # imu_ang_vel       [rad/s]
proj_g     = np.array([0.0, 0.0, -1.0])     # projected_gravity (정자세)
cmd_twist  = np.array([0.5, 0.0, 0.0])      # (v_x, v_y, ω_z)   [m/s, rad/s]
phase      = np.array([0.0, 1.0])           # (sin, cos), period 0.6 s 의 t=0

# 12 관절 순서: FL_h FL_t FL_c | FR_h FR_t FR_c | RL_h RL_t RL_c | RR_h RR_t RR_c
q_default  = np.array([0.0, 0.9, -1.8] * 4) # home 포즈 (xml keyframe "home")
q_full     = q_default.copy()                # 현재 자세 = home
q_dot_full = np.zeros(12)                    # 정지 상태
last_action = np.zeros(12)                   # 직전 raw action (k=0 가정)

joint_pos_rel = q_full - q_default           # joint_pos_rel obs
joint_vel_rel = q_dot_full                   # joint_vel_rel obs

obs = np.concatenate([
    ang_vel, proj_g, cmd_twist, phase,
    joint_pos_rel, joint_vel_rel, last_action,
])
assert obs.shape == (47,)

# ── 3. 정책 forward: a_k = π_θ(o_k) ─────────────────────────────────────────
obs_t = torch.from_numpy(obs).float()
obs_n = (obs_t - obs_mean) / obs_std         # obs_normalizer
with torch.no_grad():
    action_raw = mlp(obs_n.unsqueeze(0))[0].numpy()

# ── 4. q_des = q_default + scale * action ───────────────────────────────────
# env.yaml: actions.joint_pos.scale=0.25, use_default_offset=True
ACTION_SCALE = 0.25
q_des_full = q_default + ACTION_SCALE * action_raw

print("관측 요약 (47 dims):")
print(f"  ang_vel       = {ang_vel}")
print(f"  proj_g        = {proj_g}")
print(f"  cmd (vx,vy,ω) = {cmd_twist}")
print(f"  phase (s,c)   = {phase}")
print(f"  joint_pos_rel = {joint_pos_rel}")
print(f"  joint_vel_rel = {joint_vel_rel}")
print(f"  last_action   = {last_action}")
print()
print(f"정책 출력 a_k (12 dims) = {action_raw.round(5)}")
print(f"q_des     (12 dims)     = {q_des_full.round(5)}")
print()

# ── 5. 본 노트북은 FR (인덱스 3,4,5) 3 관절 축소 예제만 사용 ──────────────────
fr_idx = slice(3, 6)
q          = q_full[fr_idx]
q_dot      = q_dot_full[fr_idx]
q_des_prev = q_default[fr_idx]               # 직전 step 의 q_des = home 라고 가정
q_des      = q_des_full[fr_idx]
v_body     = np.array([0.5, 0.0, 0.0])       # 동체 선속도 (정책 obs 에는 없지만 4.1 에서 참고)
omega_body = ang_vel
g_proj     = proj_g
v_cmd      = cmd_twist

print("FR (FR_hip, FR_thigh, FR_calf) 3 관절 축소:")
print(f"  q          = {q}")
print(f"  q_dot      = {q_dot}")
print(f"  q_des_prev = {q_des_prev}")
print(f"  q_des      = {q_des}")


### (C) 중간 결과

정책이 한 번 계산한 $q_{\mathrm{des},k}$ 는 다음 정책 갱신 전까지 상수로 유지된다.
수식으로 쓰면, 정책 시각 $t_k$ 이후 한 정책 주기 동안

$$
q_{\mathrm{des}}(t) = q_{\mathrm{des},k},
\qquad t \in [t_k, t_k + 20\,\mathrm{ms})
$$

기호:
- $q_{\mathrm{des}}(t)$: 실제 시간 $t$ 에 PD 제어기가 참조하는 목표 관절각
- $q_{\mathrm{des},k}$: 정책 시각 $t_k$ 에 한 번 계산된 목표 관절각
- $t_k$: $k$번째 정책 갱신 시각
- $[t_k, t_k + 20\,\mathrm{ms})$: 다음 정책 갱신 전까지의 20 ms 구간

이다. 위 수식은 아래 코드에서 `q_des = np.array([...])` 로 한 번 값을 정하고,
뒤의 네 번 PD 계산이 같은 `q_des` 값을 참조하는 방식으로 구현했다.


## 3. PD 제어 단계 (5 ms)

### (A) 소스 발췌 — actuator cfg + 값 유지 분기 + 토크 계산

```python
# 출처: src/assets/robots/unitree_go2/go2_constants.py:342-358
# Method A (BE consistent: integrator/Schur/Force 전부 β_be = 1/(1+h/τ)).
_MA_MOTOR = dict(Kt=0.128, Ke=0.128, R=0.3, L=1e-4, gear_ratio=6.33,
                 substeps=_COUPLED_SUBSTEPS, pd_substeps=_PD_RECOMPUTE,
                 use_coupled=True, method="A")
GO2_METHODA_HIP = NativeElectricActuatorCfg(
  target_names_expr=(".*hip_.*",), stiffness=20.0, damping=1.0,
  effort_limit=23.5, saturation_effort=23.5, velocity_limit=30.0, armature=0.01, **_MA_MOTOR)
GO2_METHODA_THIGH = NativeElectricActuatorCfg(
  target_names_expr=(".*thigh_.*",), stiffness=20.0, damping=1.0,
  effort_limit=23.5, saturation_effort=23.5, velocity_limit=30.0, armature=0.01, **_MA_MOTOR)
GO2_METHODA_CALF = NativeElectricActuatorCfg(
  target_names_expr=(".*calf_.*",), stiffness=40.0, damping=2.0,
  effort_limit=45.0, saturation_effort=45.0, velocity_limit=30.0, armature=0.02, **_MA_MOTOR)
```

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:497-540
pd_period = cfg.pd_substeps if cfg.pd_substeps > 0 else cfg.substeps
recompute_pd = (cfg.substeps <= 1
                or self._sub_idx == 0
                or (cfg.pd_substeps > 0 and self._sub_idx % pd_period == 0))

if recompute_pd:
    # PD + DC motor saturation → τ_des
    tau_des = super().compute(cmd)
    ...
    I_des = tau_des / self._Ktgr

    # 캐시 저장
    if cfg.substeps > 1:
        self._I_des_hold = I_des
        self._tau_des_hold = tau_des
else:
    # PD 주기 사이: 캐시된 I_des 사용 (ZOH)
    I_des = self._I_des_hold
    tau_des = self._tau_des_hold
```

코드상 명칭 ↔ 본문 기호:
- `tau_des` ↔ τ_des
- `self._Ktgr` ↔ Kt · gr
- `cfg.pd_substeps = 50` ↔ PD 재계산 주기 = 50 × 0.1 ms = **5 ms**

PD 재계산 시각을 $t_m$ 이라고 하면, 아래 소스는 다음 수식을 구현한다.

$$
(\tau_{\mathrm{des}}, I_{\mathrm{des}})(t) =
\begin{cases}
\mathrm{compute\_pd\_and\_motor}(q_{\mathrm{des}}, q, \dot q), & t = t_m \\
(\tau_{\mathrm{des}}, I_{\mathrm{des}})(t_m), & t_m < t < t_{m+1}
\end{cases}
$$

기호:
- $\tau_{\mathrm{des}}$: PD와 motor saturation 이후의 목표 토크
- $I_{\mathrm{des}}$: 목표 토크를 $K_t g_r$ 로 나눈 목표 전류
- $t$: 현재 물리 substep 시각
- $t_m$, $t_{m+1}$: 연속한 두 PD 재계산 시각
- $q_{\mathrm{des}}$, $q$, $\dot q$: 목표 관절각, 현재 관절각, 현재 관절속도
- $\mathrm{compute\_pd\_and\_motor}(\cdot)$: `super().compute(cmd)` 로 대표되는 PD 및 motor saturation 계산

수식의 첫 줄은 `recompute_pd` 분기에서 `super().compute(cmd)` 와
`I_des = tau_des / self._Ktgr` 로 계산하고, 둘째 줄은 `_tau_des_hold`,
`_I_des_hold` 에 저장한 값을 다시 읽는 방식으로 구현되어 있다.

`super().compute(cmd)` 본문 = PD ( `mjlab/actuator/pd_actuator.py:96-107` `IdealPdActuator.compute` )
→ DC 모터 속도 saturation ( `mjlab/actuator/dc_actuator.py:136-162` `DcMotorActuator._clip_effort` ).
정확한 식은 8.2-8.3 정리 참조. 아래 (B) 셀은 그 식을 그대로 옮겼다.


### (B) 값 대입 — 1관절 (FR_hip) 축소 예제

saturation 식은 `DcMotorActuator._clip_effort`
( `mjlab/actuator/dc_actuator.py:136-162` ) 를 그대로 옮긴다.
이 절의 값 대입 코드는 다음 수식을 순서대로 구현한다.

$$
\tau_{\mathrm{pd}} = K_p(q_{\mathrm{des}} - q) - K_d\dot q
$$

DC motor saturation envelope (cf. 8.3):

$$
\omega_c = \omega_{\max}\,\bigl(1 + \tau_{\mathrm{limit}}/\tau_{\mathrm{sat}}\bigr),
\qquad
\tilde\omega = \mathrm{clip}(\dot q, -\omega_c, +\omega_c)
$$

$$
\tau_{\mathrm{top}}(\tilde\omega) = \tau_{\mathrm{sat}}\bigl(1 - \tilde\omega/\omega_{\max}\bigr),
\qquad
\tau_{\mathrm{bot}}(\tilde\omega) = \tau_{\mathrm{sat}}\bigl(-1 - \tilde\omega/\omega_{\max}\bigr)
$$

$$
\tau_{\max}(\tilde\omega) = \min\bigl(\tau_{\mathrm{top}},\,+\tau_{\mathrm{limit}}\bigr),
\qquad
\tau_{\min}(\tilde\omega) = \max\bigl(\tau_{\mathrm{bot}},\,-\tau_{\mathrm{limit}}\bigr)
$$

$$
\tau_{\mathrm{des}} = \mathrm{clip}\bigl(\tau_{\mathrm{pd}},\,\tau_{\min}(\tilde\omega),\,\tau_{\max}(\tilde\omega)\bigr),
\qquad
I_{\mathrm{des}} = \frac{\tau_{\mathrm{des}}}{K_t g_r}
$$

기호:
- $\tau_{\mathrm{pd}}$: stiffness/damping 으로 계산한 raw PD 토크
- $K_p$, $K_d$: 관절별 stiffness, damping gain
- $q_{\mathrm{des}}$, $q$, $\dot q$: 목표 관절각, 현재 관절각, 현재 관절속도
- $\tau_{\mathrm{sat}}$: stall (zero-speed) torque (`saturation_effort`)
- $\omega_{\max}$: no-load speed (`velocity_limit`)
- $\tau_{\mathrm{limit}}$: continuous-rating effort limit (`effort_limit`)
- $\omega_c$: torque-speed 직선이 $\tau_{\mathrm{limit}}$ 와 만나는 corner velocity
- $\tilde\omega$: corner 안쪽으로 클립된 속도 (envelope monotonicity 보장용)
- $\tau_{\max}$, $\tau_{\min}$: 현재 속도에서 허용되는 위·아래 토크 한계 (linear envelope ∩ effort_limit)
- $\mathrm{clip}(x, a, b)$: 값 $x$ 를 구간 $[a,b]$ 안으로 제한하는 함수
- $I_{\mathrm{des}}$: 목표 전류, $K_t$: torque constant, $g_r$: gear ratio

아래 코드의 `tau_pd`, `vel_clipped`, `tau_top`, `tau_bot`, `max_eff`, `min_eff`, `tau_des`, `I_des` 가 위 수식의 항에 그대로 대응한다.


In [ ]:
# FR_hip — 위 셀 (cell 6) 의 q_des, q, q_dot 를 그대로 받아 PD/saturation 계산
# (이 값을 바꿔보고 싶으면 cell 6 의 obs/action 또는 아래 gain 만 바꿔도 된다)
Kp                = 20.0     # [N·m/rad]   stiffness
Kd                = 1.0      # [N·m·s/rad] damping
effort_limit      = 23.5     # [N·m]       τ_limit  (DcMotorActuatorCfg.effort_limit)
saturation_effort = 23.5     # [N·m]       τ_sat   (DcMotorActuatorCfg.saturation_effort)
velocity_limit    = 30.0     # [rad/s]     ω_max   (DcMotorActuatorCfg.velocity_limit)

# FR (인덱스 3,4,5) 중 첫 관절 = FR_hip
q_des_j = q_des[0]    # 정책에서 받은 FR_hip 목표각 (cell 6)
q_j     = q[0]        # 현재 q
q_dot_j = q_dot[0]    # 현재 q̇

# (1) PD: τ_pd = Kp·(q_des - q) - Kd·q̇   (mjlab/actuator/pd_actuator.py:96-107)
tau_pd = Kp * (q_des_j - q_j) - Kd * q_dot_j
print(f"τ_pd = {tau_pd:.4f} [N·m]   (q_des={q_des_j:.4f}, q={q_j:.4f}, q̇={q_dot_j:.4f})")

# (2) DC motor saturation (mjlab/actuator/dc_actuator.py:136-162)
#     1) corner velocity ω_c
#     2) ω̃ = clip(q̇, ±ω_c)
#     3) 위/아래 직선 envelope τ_top, τ_bot
#     4) effort_limit 로 한 번 더 클램프 → τ_max, τ_min
#     5) 최종 τ_des = clip(τ_pd, τ_min, τ_max)
omega_c     = velocity_limit * (1.0 + effort_limit / saturation_effort)
vel_clipped = max(-omega_c, min(omega_c, q_dot_j))
tau_top     = saturation_effort * (1.0  - vel_clipped / velocity_limit)
tau_bot     = saturation_effort * (-1.0 - vel_clipped / velocity_limit)
max_eff     = min(tau_top,  +effort_limit)
min_eff     = max(tau_bot,  -effort_limit)
tau_des     = max(min_eff, min(max_eff, tau_pd))
print(f"ω_c = {omega_c:.4f} [rad/s],  ω̃ = {vel_clipped:.4f}")
print(f"τ_top = {tau_top:.4f},  τ_bot = {tau_bot:.4f}")
print(f"τ_max = {max_eff:.4f},  τ_min = {min_eff:.4f}")
print(f"τ_des = {tau_des:.4f} [N·m]")

# (3) 토크 → 목표 전류 (mj_native_electric_actuator.py:531)
Kt, gr  = 0.128, 6.33
Kt_gr   = Kt * gr
I_des   = tau_des / Kt_gr
print(f"I_des = τ_des / (Kt·gr) = {I_des:.4f} [A]   (Kt·gr = {Kt_gr:.4f})")


### (C) 중간 결과

PD 단계에서 계산한 값은 다음 50 개의 물리 substep 동안

$$
\tau_{\mathrm{des}}^{(r)} = \tau_{\mathrm{des}}(t_m),
\qquad
I_{\mathrm{des}}^{(r)} = I_{\mathrm{des}}(t_m),
\qquad r = 0, \dots, 49
$$

기호:
- $r$: 하나의 PD 주기 안에서의 물리 substep index
- $\tau_{\mathrm{des}}^{(r)}$, $I_{\mathrm{des}}^{(r)}$: $r$번째 물리 substep 에서 사용하는 목표 토크와 목표 전류
- $t_m$: 해당 PD 주기가 시작될 때의 PD 재계산 시각
- $0,\dots,49$: 5 ms PD 주기 안의 50개 물리 substep

로 유지된다. 위 수식은 코드에서 `_tau_des_hold`, `_I_des_hold` 에 저장한 값을
각 substep 에서 다시 읽는 방식으로 구현되어 있다. 물리 단계는 매 substep 마다
최신 $\omega$ 로 가상 전압 $V$ 만 갱신한다
(`mj_native_electric_actuator.py:542-557`).


## 4. 물리 단계 (0.1 ms)

한 substep 안에서 일어나는 계산을 4 개 절로 분해한다.


### 4.1 운동방정식

#### (A) 소스 발췌 — actuator force = Kt·gr·I

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:14-19, 33-46
#   d->act[i]  = 전류 I  [A]
#   d->ctrl[i] = 제어 입력 (filterexact: 등가전류, user: 전압)
#   force      = gainprm[0] × act = Kt·gr × I
```

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:351-353
# ── Gain: force = Kt·gr × I ─────────────────────────
act.gaintype = mujoco.mjtGain.mjGAIN_FIXED
act.gainprm[0] = self._Ktgr
```

운동방정식 자체 (M, C, g, J_c) 의 어셈블리 본문은 mujoco_warp 본체와
mujoco fork (C) 양쪽에 있다 — 코드 위치는 §7 참조표 참고.
본 환경의 site-packages 에서 직접 확인 가능:
`/home/rbdo/miniconda3/envs/mjlab/lib/python3.11/site-packages/mujoco_warp/_src/`.

본 절에서는 표준형

$$
M(q)\,\ddot q \;+\; b\dot q \;+\; C(q,\dot q) \;+\; g(q)
\;=\; \tau_{\text{actuator}} \;+\; J_c^{\!\top}\,\lambda,
\qquad \tau_{\text{actuator}} = K_t\,g_r\,I
$$

또는 우변 force 형태로 옮기면

$$
M(q)\,\ddot q
= \tau_{\text{actuator}} - b\dot q - C(q,\dot q) - g(q) + J_c^{\!\top}\lambda
$$

이다.

기호:
- $M(q)$: 관절 위치 $q$ 에서의 mass matrix
- $q$, $\dot q$, $\ddot q$: 관절 위치, 관절속도, 관절가속도
- $b\dot q$: viscous damping 일반화 힘
- $C(q,\dot q)$: Coriolis/centrifugal 항을 포함한 속도 의존 일반화 힘
- $g(q)$: 중력 일반화 힘
- $\tau_{\text{actuator}}$: actuator 가 관절에 가하는 토크
- $J_c$: contact Jacobian, $J_c^\top\lambda$ 는 접촉력이 관절공간으로 투영된 항
- $\lambda$: contact force 또는 constraint impulse 계열의 미지수
- $K_t$: torque constant, $g_r$: gear ratio, $I$: motor 전류

만 가정하고 진행한다. 위 수식의 $\tau_{\text{actuator}} = K_t g_r I$ 는
소스에서 `act.gainprm[0] = self._Ktgr` 로 $K_t g_r$ 를 gain 에 넣고,
MuJoCo 가 `force = gainprm[0] * act` 를 계산하는 방식으로 구현되어 있다.


#### (B) 값 대입 — 자유도 2 축소 모형

접촉항을 잠시 빼면 위 운동방정식은 다음 계산으로 축소된다.

$$
\ddot q_{\mathrm{free}} = M(q)^{-1}\left(\tau_{\mathrm{actuator}} - b\dot q - C(q,\dot q) - g(q)\right)
$$

기호:
- $\ddot q_{\mathrm{free}}$: 접촉 constraint 를 아직 적용하지 않은 비구속 관절가속도
- $M(q)^{-1}$: mass matrix 의 역행렬
- $\tau_{\mathrm{actuator}}$: actuator 토크 벡터
- $b\dot q$: viscous damping 토크 벡터
- $C(q,\dot q)$, $g(q)$: 각각 속도 의존 일반화 힘과 중력 일반화 힘

아래 코드는 이 수식을 `M_inv @ (tau_actuator - damping_qdot - C_qdot - g_q)` 로 구현한다.

In [ ]:
import numpy as np

# 자유도 2 축소 모형 (이 값을 바꿔보세요)
M = np.array([[1.5, 0.1],
              [0.1, 0.8]])              # 질량 행렬 [kg·m²]
q_dot = np.array([0.20, -0.10])          # 관절속도 [rad/s]
damping_b = np.array([0.30, 0.20])       # viscous damping [N·m·s/rad]
damping_qdot = damping_b * q_dot         # b · q̇ [N·m]
C_qdot = np.array([0.05, -0.02])         # Coriolis · q̇ [N·m]
g_q    = np.array([3.00,  0.50])         # 중력 토크   [N·m]

# actuator 토크 (이미 Kt·gr·I 형태로 들어왔다고 가정)
tau_actuator = np.array([4.0, 1.5])      # [N·m]

# 비구속 가속도 (접촉 무시)
M_inv = np.linalg.inv(M)
q_ddot_free = M_inv @ (tau_actuator - damping_qdot - C_qdot - g_q)

print(f"M     =\n{M}")
print(f"M^-1  =\n{M_inv}")
print(f"damping b·q̇ = {damping_qdot}")
print(f"비구속 가속도 q̈_free = {q_ddot_free}")


#### (C) 중간 결과

q̈_free 가 4.2 의 입력. 접촉이 있으면 4.2 에서 λ 가 더해져 q̈ 가 보정된다.


### 4.2 β_imp — Schur complement / Force RHS

#### (A) 소스 발췌 — Schur 항 (qDeriv 누적용) + Force RHS 보정

```python
# 출처: mujoco_warp/_src/derivative.py:73-89 — 본 노트북은 dynprm[4]=0 분기만 다룸
# Schur term: -(1-β_imp)·Kt·gr·Ke·gr/R for FILTEREXACT motor coupling.
# Method A: 1-β_imp = h/(τ+h)  (BE)
schur = float(0.0)
if actuator_dyntype[actid] == DynType.FILTEREXACT:
    dynprm_act = actuator_dynprm[actuator_dynprm_id, actid]
    Ke_gr = dynprm_act[1]
    L_val = dynprm_act[2]
    if Ke_gr != 0.0 and L_val > MJ_MINVAL:
        tau_e = wp.max(MJ_MINVAL, dynprm_act[0])
        Kt_gr = actuator_gainprm[actuator_gainprm_id, actid][0]
        h_dt = opt_timestep[worldid % opt_timestep.shape[0]]
        one_minus_beta = h_dt / (tau_e + h_dt)
        R_val = L_val / tau_e
        schur = -one_minus_beta * Kt_gr * Ke_gr / R_val
```

```python
# 출처: mujoco_warp/_src/derivative.py:170, 209-217
# qDeriv 누적과 M_eff = qM − h · qDeriv
qderiv_contrib += moment_i * moment_j * vel
...
qderiv *= opt_timestep[worldid % opt_timestep.shape[0]]
qM_out = qM_in - qderiv         # M_eff = qM − h·qDeriv
```

```python
# 출처: mujoco_warp/_src/forward.py:765-787 (kernel _actuator_force)
# Method A RHS correction (Schur complement RHS):
#   force currently uses I_old (act_in[act_last]) when not actearly.
#   Method A predicts I_new = β·I_old + (1-β)·ctrl  with  β = 1/(1+h/τ).
#   ΔF = gain·(1-β)·(ctrl - I_old).
if na and act_first >= 0:
    if actuator_dyntype[uid] == DynType.FILTEREXACT:
        dynprm_uid = actuator_dynprm[worldid % actuator_dynprm.shape[0], uid]
        if dynprm_uid[1] != 0.0 and dynprm_uid[2] > MJ_MINVAL and not actuator_actearly[uid]:
            tau_e = wp.max(MJ_MINVAL, dynprm_uid[0])
            h_dt = opt_timestep[worldid % opt_timestep.shape[0]]
            one_minus_beta = h_dt / (tau_e + h_dt)
            I_old = act_in[worldid, act_last]
            force += gain * one_minus_beta * (ctrl - I_old)
```

코드상 명칭 ↔ 본문 기호:
- `one_minus_beta` ↔ $1 - \beta_{\text{imp}}$
- `schur` ↔ Schur 항 $-(1-\beta_{\text{imp}})\,K_t g_r K_e g_r / R$
- `qderiv` (누적, h-스케일 후) ↔ $h \cdot J_c^{\!\top} B J_c$ 의 한 원소
- `force` (after `+= gain · (1−β_imp)·(ctrl−I_old)`) ↔ Force RHS 보정 후 토크

Method A 에서 이 절의 소스가 구현하는 핵심 수식은 다음 두 개다.

#### 물리적 block matrix

한 step $[t,\,t+h]$ 동안 좌변의 $\Delta\dot q$, $\Delta i$ 가 한꺼번에 결정된다고
(즉 step 종료 시점 양에 대한 force balance 를) 둔 형태:

$$
\begin{bmatrix}
\dfrac{M}{h} + \Bigl(b + c + \dfrac{\partial c}{\partial \dot q}\,\dot q\Bigr) & -K_t g_r \\[1.2em]
K_e g_r & \dfrac{L}{h} + R
\end{bmatrix}
\begin{bmatrix}\Delta \dot q \\[0.4em] \Delta i\end{bmatrix}
=
\begin{bmatrix}
K_t g_r\,i_k - b\dot q - c\dot q - G + \tau_{\mathrm{applied}} + \tau_{\mathrm{constraint}} \\[0.4em]
V - R\,i_k - K_e g_r\,\dot q_k
\end{bmatrix}
$$

기호:
- $M(q)$: mass matrix, $b$: joint damping, $c(q,\dot q)$: Coriolis matrix, $G(q)$: 중력
- $\partial c/\partial \dot q$: Coriolis matrix 의 속도 Jacobian
- $K_t g_r$: torque constant × gear ratio (전류 → 토크), $K_e g_r$: back-EMF × gear ratio
- $L$, $R$: winding inductance, resistance, $V$: motor 인가전압
- $h$: 물리 적분 시간 간격 (`m.opt.timestep`)
- $\dot q_k$, $i_k$: 현재 step 의 관절속도, 전류, $\Delta\dot q$, $\Delta i$: 한 step 동안의 증분
- $\tau_{\mathrm{applied}}$, $\tau_{\mathrm{constraint}}$: 외력·접촉으로 들어오는 일반화 힘

abstract 표기 $\begin{bmatrix}A & B \\ C & D\end{bmatrix}$ 와 일대일 대응:

$$
A = \frac{M}{h} + B_{\mathrm{mech}},\quad
B = -K_t g_r,\quad
C = K_e g_r,\quad
D = \frac{L}{h} + R
$$

여기서 $B_{\mathrm{mech}} \equiv b + c + (\partial c/\partial\dot q)\dot q$.

블록 우변의 두 항을 이름붙여 둔다 (아래 Schur 전개 내내 사용):

$$
F_{\mathrm{mech}} \;\equiv\; K_t g_r\,i_k \;-\; b\dot q \;-\; c\dot q \;-\; G
\;+\; \tau_{\mathrm{applied}} \;+\; \tau_{\mathrm{constraint}}
$$

$$
F_{\mathrm{elec}} \;\equiv\; V \;-\; R\,i_k \;-\; K_e g_r\,\dot q_k
$$

- $F_{\mathrm{mech}}$: 현재 step 시작 시점의 전류 $i_k$ 만으로 계산한 actuator torque
  ($K_t g_r\,i_k$) + 외력/접촉/속도성 일반화 힘. 즉 좌변이 0 일 때 (모터 coupling
  무시) $\Delta\dot q$ 를 만들어 내는 우변.
- $F_{\mathrm{elec}}$: 인가전압에서 저항강하·역기전력을 뺀 잉여전압. 이게 0 이면
  전류는 변하지 않고 ($\Delta i = 0$), 이게 양수면 전류가 늘어난다.

#### $\Delta i$ 를 제거하는 Schur complement 전개

(i) 두 번째 행에서 $\Delta i$ 풀기:

$$
\Delta i \;=\; \frac{1}{L/h + R}\Bigl[\,F_{\mathrm{elec}} \;-\; K_e g_r\,\Delta\dot q\,\Bigr]
$$

(ii) 첫 행에 대입:

$$
\Bigl(\tfrac{M}{h} + B_{\mathrm{mech}}\Bigr)\Delta\dot q
\;-\; K_t g_r\cdot\frac{F_{\mathrm{elec}} - K_e g_r\,\Delta\dot q}{L/h + R}
\;=\; F_{\mathrm{mech}}
$$

(iii) $\Delta\dot q$ 항을 좌변에 모으고 $F_{\mathrm{elec}}$ 항을 우변으로:

$$
\underbrace{\Bigl[\tfrac{M}{h} + B_{\mathrm{mech}}
+ \tfrac{K_t g_r\,K_e g_r}{L/h + R}\Bigr]}_{\text{Schur complement }A - BD^{-1}C}\,\Delta\dot q
\;=\; F_{\mathrm{mech}} \;+\; \underbrace{\tfrac{K_t g_r}{L/h + R}\,F_{\mathrm{elec}}}_{\text{Force RHS 보정의 원형}}
$$

좌변 마지막 항이 §아래 `M_eff` 보정으로, 우변 마지막 항이 §아래 Force RHS 보정으로
각각 이어진다. 같은 분모 $L/h + R$ 이 두 곳에 동시에 등장하는 것이 핵심이다.

#### `qDeriv` / `M_eff` 코드 표기와의 연결

코드는 $\Delta\dot q$ 가 아니라 $\ddot q = \Delta\dot q / h$ 를 직접 푼다.
위 식 양변에 $h$ 를 곱하면 좌변이 $M_{\mathrm{eff}}\,\ddot q$ 형태가 된다:

$$
M_{\mathrm{eff}} \equiv M + h\cdot B_{\mathrm{mech}} + h\cdot\frac{K_t g_r\,K_e g_r}{L/h + R}
$$

마지막 항을 $\tau_e = L/R$ 로 정리하면 $L/h + R = R(\tau_e + h)/h$ 이므로

$$
h\cdot\frac{K_t g_r\,K_e g_r}{L/h + R}
= \frac{h^2\,K_t g_r\,K_e g_r}{R(\tau_e + h)}
= h\cdot\underbrace{\tfrac{h}{\tau_e+h}}_{=\,1-\beta_{\mathrm{imp}}}\cdot\frac{K_t g_r\,K_e g_r}{R}
$$

이게 코드의 `schur` 부호반전 형태다 (`schur = -one_minus_beta * Kt_gr * Ke_gr / R_val`).
누적/스케일 단계는 그대로 따라간다 (mujoco_warp/_src/derivative.py):

$$
\text{(per-actuator)}\quad J_c^{\!\top} J_c \cdot \mathrm{schur} \;<\;0
\;\;\xrightarrow[\text{*= h}]{\,\text{:209}\,}\;\;
q\mathrm{Deriv} = -\,h\,(1-\beta_{\mathrm{imp}})\,\tfrac{K_t g_r\,K_e g_r}{R}\,J_c^{\!\top} J_c
$$

$$
\xrightarrow[\text{:212-217}]{\;qM\,-\,qderiv\;}\;\;
M_{\mathrm{eff}} = qM - q\mathrm{Deriv}
= M + h\,(1-\beta_{\mathrm{imp}})\,\tfrac{K_t g_r\,K_e g_r}{R}\,J_c^{\!\top} J_c
$$

위 Schur complement 식의 마지막 항과 정확히 같다.

joint damping $b$ 도 같은 경로다: derivative.py:206-207 에서
`qderiv -= dof_damping[...]` (음의 기여) → `qderiv *= h` → `qM - qderiv` 로
$M_{\mathrm{eff}}$ 대각에 $+h\,b$ 가 더해진다. 즉 위 $h\cdot B_{\mathrm{mech}}$
중 $b$ 부분이 mjwarp 에서 implicit 으로 처리되는 항이다.

주의: 위 block matrix 의 $c$, $(\partial c/\partial\dot q)\dot q$ 항은 mjwarp
`IMPLICITFAST` 경로에서는 $M_{\mathrm{eff}}$ 에 들어가지 않고 RHS 의
`qfrc_bias` ($= C(q,\dot q) + G(q)$, RNE 결과) 로만 처리된다.
좌변 보정은 실제로 $b$ 와 motor Schur 항만 들어가고, Coriolis 의 implicit
처리는 별도 (`IMPLICIT` integrator) 경로에서만 일어난다.

#### Force RHS 보정의 자세한 전개

위 Schur 식 양변에 $h$ 를 곱한 우변은 다음 두 덩어리다 (좌변은 §위 `M_eff` 식):

$$
\text{RHS} \;=\; h\cdot F_{\mathrm{mech}}
\;+\; h\cdot\underbrace{\tfrac{K_t g_r}{L/h + R}\,F_{\mathrm{elec}}}_{\text{Schur RHS 보정}}
$$

(i) 우변 두 번째 덩어리의 $h\cdot K_t g_r/(L/h + R)$ 인수를 $\tau_e = L/R$ 로 정리:

$$
\frac{h\,K_t g_r}{L/h + R}
\;=\; \frac{h^2\,K_t g_r}{L + R\,h}
\;=\; \frac{h^2\,K_t g_r}{R(\tau_e + h)}
\;=\; \underbrace{\tfrac{h}{\tau_e + h}}_{=\,1-\beta_{\mathrm{imp}}}\cdot\frac{K_t g_r}{R}\cdot h
$$

따라서

$$
h\cdot\frac{K_t g_r}{L/h + R}\,F_{\mathrm{elec}}
\;=\; h\,(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r}{R}\,F_{\mathrm{elec}}
$$

이 형태가 좌변 `schur` 항과 같은 $1-\beta_{\mathrm{imp}}$ 와 같은 $K_t g_r/R$ 가
공통으로 들어가는 이유다 ($K_e g_r$ 만 빠진 자리에 $F_{\mathrm{elec}}$ 이 들어옴).

(ii) 모터 컨트롤러 가정으로 $F_{\mathrm{elec}}$ 풀기. mjwarp 의 `dynprm[4]=0` 분기에서
전류 ODE 는

$$
\frac{dI}{dt} \;=\; \frac{\mathrm{ctrl} - I}{\tau_e},
\qquad \mathrm{ctrl} \equiv i_{\mathrm{des}}
$$

이다 (`forward.py:692-704`, `act_dot = (ctrl - act)/tau_e`). 이 식이 실제 1차 회로
$L\,dI/dt = V - R\,I - K_e g_r\,\dot q$ 와 같아지려면 컨트롤러가

$$
V \;=\; R\,\mathrm{ctrl} \;+\; K_e g_r\,\dot q_k
$$

로 인가전압을 잡아 줘야 한다 (저항강하 보상 + 역기전력 보상 = ideal current servo).
이 $V$ 를 $F_{\mathrm{elec}}$ 정의에 대입:

$$
F_{\mathrm{elec}} \;=\; V - R\,i_k - K_e g_r\,\dot q_k
\;=\; (R\,\mathrm{ctrl} + K_e g_r\,\dot q_k) - R\,i_k - K_e g_r\,\dot q_k
\;=\; R\,(i_{\mathrm{des}} - i_k)
$$

(iii) (i)·(ii) 를 합치면 Schur RHS 보정 항은

$$
h\,(1-\beta_{\mathrm{imp}})\,\frac{K_t g_r}{R}\cdot R\,(i_{\mathrm{des}} - i_k)
\;=\; h\,K_t g_r\,(1-\beta_{\mathrm{imp}})\,(i_{\mathrm{des}} - i_k)
$$

즉 RHS 전체를 묶으면

$$
\text{RHS} \;=\; h\,\bigl[\underbrace{K_t g_r\,i_k}_{=\,F_{\mathrm{pre}}}
\;+\; \underbrace{K_t g_r\,(1-\beta_{\mathrm{imp}})\,(i_{\mathrm{des}} - i_k)}_{=\,\Delta F_{\mathrm{Schur RHS}}}
\;-\; b\dot q - c\dot q - G + \tau_{\mathrm{applied}} + \tau_{\mathrm{constraint}}\bigr]
$$

(iv) actuator force 부분만 모으면

$$
\boxed{\;F_{\mathrm{post}} \;=\; F_{\mathrm{pre}} \;+\; K_t g_r\,(1-\beta_{\mathrm{imp}})\,(i_{\mathrm{des}} - i_k)\;}
$$

이고, 등가 형태로 다시 쓰면

$$
F_{\mathrm{post}} \;=\; K_t g_r\bigl[\beta_{\mathrm{imp}}\,i_k + (1-\beta_{\mathrm{imp}})\,i_{\mathrm{des}}\bigr]
$$

즉 actuator force 의 인자를 "step 시작 전류 $i_k$" 가 아니라
"한 step 안에서 $\beta_{\mathrm{imp}}$ 와 $(1-\beta_{\mathrm{imp}})$ 로 가중평균한
implicit 예측 전류" 로 바꾼 것이다.

#### 코드와의 대응

`mujoco_warp/_src/forward.py:765-787` (`_actuator_force`) 의

```python
force = gain * ctrl_act + bias                      # ctrl_act = act_in[act_last] = i_k
                                                    # → force = K_t·gr · i_k = F_pre
...
force += gain * one_minus_beta * (ctrl - I_old)     # = +K_t·gr·(1-β_imp)·(i_des - i_k)
                                                    # → force = F_post
```

가 위 (iv) 박스 식을 그대로 한 줄씩 구현한다. `gain = K_t g_r` (`gainprm[0]`),
`ctrl = i_des`, `I_old = act_in[act_last] = i_k`, `one_minus_beta = h/(τ_e + h)`.

좌변 `schur` 와 우변 보정이 같은 $(1-\beta_{\mathrm{imp}})$ 인수를 공유하는 게
한 step 안에서 $i$ 와 $\dot q$ 를 일관되게 implicit 으로 묶는다 — 이 짝짓기가
본 노트북이 다루는 분기 (`dynprm[4]=0`) 의 정의다.

#### 위 두 식의 기호 정리

- $1-\beta_{\mathrm{imp}} \equiv h/(\tau_e + h)$: implicit 가중치 (본 분기 정의)
- $J_c$: actuator moment Jacobian (한 행이 한 actuator, 열이 nv DOF)
- $\mathrm{schur}$: 코드의 per-actuator 스칼라 $-(1-\beta_{\mathrm{imp}})\,K_t g_r K_e g_r / R$
- $F_{\mathrm{mech}}$, $F_{\mathrm{elec}}$: 위 정의 — 블록 우변 두 행
- $F_{\mathrm{pre}} \equiv K_t g_r\,i_k$: 보정 전 actuator torque (코드의 첫 줄 `force = gain*ctrl_act+bias`, bias=0 가정)
- $F_{\mathrm{post}}$: Schur RHS 보정 후 actuator torque (코드 두 번째 줄까지 적용한 값)
- $i_{\mathrm{des}}$, $i_k$: 목표 전류, 현재 step 시작 전류 (코드의 `ctrl`, `act_in[act_last]`)
- $V$: 모터 인가전압 (위 (ii) 의 컨트롤러 가정으로 $R\,\mathrm{ctrl} + K_e g_r\,\dot q_k$)


#### (B) 값 대입 — 1관절 Schur 1-step + Force RHS

접촉이 있는 일반 경우의 $\ddot q$ 는 closed-form 으로 떨어지지 않는다.
실제로 mjwarp 가 한 step 안에서 푸는 것은 다음 nefc 차원 convex 최적화 (Gauss
principle of least constraint) 의 최소해다:

$$
\ddot q^{\,*} \;=\; \arg\min_{\ddot q}\;
\tfrac12\,(\ddot q - \ddot q_{\mathrm{free}})^{\!\top}\,M_{\mathrm{eff}}\,
(\ddot q - \ddot q_{\mathrm{free}})
\;+\;\sum_{r=1}^{n_{\mathrm{efc}}}\;\mathrm{cost}_r\!\bigl((J_c\ddot q - a_{\mathrm{ref}})_r\bigr)
$$

기호:
- $\ddot q_{\mathrm{free}}$: §4.1 의 비구속 가속도 (`d.qacc_smooth`)
- $M_{\mathrm{eff}}$: §4.2 의 Schur 보정된 mass matrix
- $J_c$ (`d.efc.J`): nefc × nv constraint Jacobian (equality, joint-limit, friction, normal contact 행 모두)
- $a_{\mathrm{ref}}$ (`d.efc.aref`): Baumgarte 안정화 reference acceleration
- $\mathrm{cost}_r(\cdot)$: 행 $r$ 의 type 에 따른 단변량 비용
  (등식: 이차, 부등식·friction: barrier/penalty; `solref`/`solimp` 로 매끈화)
- $n_{\mathrm{efc}}$: 활성 constraint 행 수 (`d.nefc`)

목적함수는 $\ddot q$ 에 대해 strictly convex 하므로
(regularization $\mathrm{diag}(\mathrm{efc\_D})>0$ 덕분에) PCG 또는
Newton-with-linesearch 가 수렴한다 (`mujoco_warp/_src/solver.py`,
`solve` → `_solve` → `_solver_iteration`, `m.opt.iterations` 만큼). 수렴 후
$\lambda \equiv \mathrm{efc\_force}$,  $\tau_{\mathrm{constraint}} = J_c^{\!\top}\lambda$
(코드 위치는 §7 참조표).

따라서 일반적으로 1자유도 closed-form $\lambda = b/A$ 같은 식은 성립하지 않는다.
본 셀은 hip 관절 (FR_hip) 만 다루므로 직접 contact constraint 가 없고,
$\ddot q = \ddot q_{\mathrm{free}}$ 로 둔다.

좌변 Schur (코드 명칭과 동일):

$$
q\mathrm{Deriv} = h\,J_c^\top \cdot \mathrm{schur} \cdot J_c,
\qquad
M_{\mathrm{eff}} = M - q\mathrm{Deriv}
$$

$1-\beta_{\mathrm{imp}} = h/(\tau_e+h) > 0$ 이고 $K_t g_r,\,K_e g_r,\,R > 0$ 이므로
$\mathrm{schur} = -(1-\beta_{\mathrm{imp}})\,K_t g_r K_e g_r / R < 0$,
따라서 $q\mathrm{Deriv}<0$, $M_{\mathrm{eff}}>M$ (양정정성 강화).
`M_eff` 표기는 §위 block matrix 풀이의 좌변 Schur complement 를 mass matrix 쪽 이름으로 적은 것.

비구속 가속도:

$$
\ddot q_{\mathrm{free}} = M_{\mathrm{eff}}^{-1}\bigl(F_{\mathrm{post}} - b\dot q - C_q - g_q\bigr)
$$

기호:
- $q\mathrm{Deriv}$, $M_{\mathrm{eff}}$: 위 Schur 전개의 좌변 보정 (코드 명칭 그대로)
- $J_c$: actuator moment Jacobian (이 셀에서는 단순화 위해 1)
- $\mathrm{schur}$: 위 Schur 식의 좌변 음의 보정항 (코드의 `schur`)
- $F_{\mathrm{post}}$: Force RHS 보정 후 actuator force, 코드의 `force`
- $b\dot q$, $C_q$, $g_q$: viscous damping, Coriolis, 중력 일반화 힘
- $\ddot q_{\mathrm{free}}$: 접촉 보정 전 비구속 가속도 (`qacc_smooth`)
- $\ddot q$: 접촉 보정 후 가속도 — FR_hip 자체엔 직접 contact 가 없어 본 셀에서는 $\ddot q = \ddot q_{\mathrm{free}}$

코드의 `qderiv`, `M_eff`, `qddot_free`, `qddot` 가 위 항들에 대응한다.

In [ ]:
import numpy as np

# 모터·통합 파라미터 (이 값을 바꿔보세요)
Kt, Ke, gr = 0.128, 0.128, 6.33
R, L       = 0.3, 1e-4
h          = 1e-4              # physics dt = 0.1 ms

tau_e   = L / R                 # 전기 시정수
Kt_gr   = Kt * gr
Ke_gr   = Ke * gr

# β_imp (Method A: dynprm[4]=0 → BE 분기)
one_minus_beta_imp = h / (tau_e + h)
beta_imp           = 1.0 - one_minus_beta_imp
print(f"τ_e        = {tau_e*1e6:.2f} µs")
print(f"β_imp      = {beta_imp:.6f}")
print(f"1 − β_imp  = {one_minus_beta_imp:.6f}")

# Schur scalar (per actuator)
schur = -one_minus_beta_imp * Kt_gr * Ke_gr / R
print(f"schur      = {schur:.6f}")

# 1자유도 단순화: J_c (actuator moment) = 1
J_c_act = 1.0
qderiv_pre = J_c_act * J_c_act * schur     # _qderiv_actuator_passive_actuation_sparse
qderiv     = h * qderiv_pre                # _qderiv_actuator_passive 의 *=h
print(f"qderiv     = {qderiv:.6e}")

# M_eff = M − h·qDeriv (식의 M_eff 자체는 qM − qderiv 로 표기됨, qderiv 가 이미 h 스케일)
M_scalar = 0.5                              # [kg·m²] (이 값을 바꿔보세요)
M_eff    = M_scalar - qderiv                # 양수 qderiv 면 M_eff 가 더 작아지지만,
                                             # Method A 에서는 schur < 0 이므로 qderiv < 0
                                             # → M_eff > M (양정정 강화)
print(f"M = {M_scalar:.6f},  M_eff = {M_eff:.6f}")

# Force RHS 보정
I_old = 1.0                                 # [A]   직전 step 전류
ctrl  = 5.0                                 # [A]   target 전류 (= I_des)
gain  = Kt_gr                               # gainprm[0]
force_pre  = gain * I_old
force_post = force_pre + gain * one_minus_beta_imp * (ctrl - I_old)
print(f"force pre  = {force_pre:.6f} [N·m]")
print(f"force post = {force_post:.6f} [N·m]   (= Schur RHS 보정된 토크)")

# 비구속 가속도 (접촉 무시) — 4.1 의 같은 식을 1자유도로
q_dot_scalar = 0.20                          # [rad/s]
damping_b    = 0.30                          # [N·m·s/rad]
damping_force = damping_b * q_dot_scalar     # b · q̇ [N·m]
C_q   = 0.0
g_q_v = 2.0                                  # [N·m]
qddot_free = (force_post - damping_force - C_q - g_q_v) / M_eff
print(f"damping b·q̇ = {damping_force:.6f} [N·m]")
print(f"q̈_free    = {qddot_free:.6f}")

# closed-form λ = b/A 는 일반적으로 성립하지 않으므로 제거.
# 실제 contact 풀이는 mujoco_warp/_src/solver.py 의 nefc 차원 convex 최적화로
# 풀린다 (코드 위치는 §7 참조표). 본 셀은 FR_hip (hip 관절, 발이 아님) 만
# 다루므로 직접 contact constraint 행이 없어 비구속 가속도를 그대로 넘긴다.
qddot = qddot_free
print(f"q̈ = {qddot:.6f}   (FR_hip 직접 contact 없음 → q̈_free 와 같음)")


#### (C) 중간 결과

이 절의 핵심은 두 가지 β_imp 사용처를 한 번에 보여준 것이다.

1. **좌변 (M_eff):** `qDeriv += −(1−β_imp)·Kt·gr·Ke·gr/R · J_cᵀJ_c` →
   `M_eff = qM − h·qDeriv` (`derivative.py`). 여기서 Schur 는 위 block matrix 풀이의
   $-BD^{-1}C$ ($\;= -h\,(1-\beta_{\mathrm{imp}})\,K_t g_r K_e g_r / R$ to leading order in $h$) 이므로
   음의 부호를 포함한다.
2. **우변 (force):** actuator 쪽 RHS 는 `force += gain·(1−β_imp)·(ctrl − I_old)` (`forward.py`) 로 보정되고,
   전체 운동방정식 RHS 에서는 추가로 `−b·q_dot − C_q − g_q` 를 뺀다.

좌·우변 모두 같은 $1-\beta_{\mathrm{imp}} = h/(\tau_e+h)$ 를 사용하는 것이
본 노트북이 다루는 분기의 정의 (`dynprm[4] = 0`) 이며,
그 결과 풀린 비구속 가속도 $\ddot q_{\mathrm{free}}$ 가
(접촉이 있으면) §4.2 끝에 인용한 nefc 차원 convex 최적화로 풀려 최종 $\ddot q$ 가 되고,
그게 §4.3 의 입력이 된다.

기호:
- $\tau_e$: 전기 시정수 ($L/R$)
- $\ddot q_{\mathrm{free}}$: 비구속 가속도 (`qacc_smooth`)
- $\ddot q$: 위 최적화로 접촉 보정까지 끝난 관절가속도. FR_hip 만 다루는 본 셀에서는 $\ddot q = \ddot q_{\mathrm{free}}$


### 4.3 β_int — 적분 단계

#### (A) 소스 발췌 — β_int 적분기 + act_dot

```python
# 출처: mujoco_warp/_src/forward.py:147-170 — 본 노트북은 dynprm[4]=0 분기만 다룸
if actuator_dyntype == DynType.FILTEREXACT:
    tau = wp.max(MJ_MINVAL, actuator_dynprm[0])
    # Motor coupling detection: dynprm[1]=Ke*gr != 0 AND dynprm[2]=L > 0
    # Method A: β_int = 1/(1+h/τ)  (BE)
    if actuator_dynprm[1] != 0.0 and actuator_dynprm[2] > MJ_MINVAL:
        act = act_in + act_dot_scale * act_dot_in * opt_timestep / (1.0 + opt_timestep / tau)
```

```python
# 출처: mujoco_warp/_src/forward.py:692-704 (kernel _actuator_force, FILTEREXACT 분기)
elif dyntype == DynType.FILTEREXACT:
    # Coupled filterexact: standard filter + Ke mismatch correction.
    #   dI/dt = (ctrl - act) / tau  +  (Ke_nom*gr - Ke_plant*gr) * omega / L
    act = act_in[worldid, act_last]
    tau_e = wp.max(MJ_MINVAL, dynprm[0])
    act_dot = (ctrl - act) / tau_e
    L_dyn = dynprm[2]
    if L_dyn > MJ_MINVAL:
        # omega at step start; ZOH over dt (kernel picks latest actuator_velocity).
        omega = actuator_velocity_in[worldid, uid]
        act_dot += (dynprm[3] - dynprm[1]) * omega / L_dyn
```

코드상 명칭 ↔ 본문 기호:
- `act_in`, `act_out` (또는 갱신된 `act`) ↔ $I_n$, $I_{n+1}$
- `act_dot_in` (= 위에서 계산된 `act_dot`) ↔ $\dot I$
- `opt_timestep / (1 + opt_timestep / tau)` ↔ Method A 적분 식의
  $h \cdot \dot I / (1 + h/\tau) = (1 - \beta_{\text{int}})\,\tau\,\dot I$
  ($\beta_{\text{int}} = 1/(1+h/\tau)$).

`filterexact` 분기에서 전류 미분은 다음 수식으로 계산된다.

$$
\dot I = \frac{I_{\mathrm{des}} - I_n}{\tau_e}
+ \frac{(K_{e,\mathrm{nom}}g_r - K_{e,\mathrm{plant}}g_r)\omega}{L}
$$

Method A 의 전류 적분은 다음 수식이다.

$$
I_{n+1} = I_n + \frac{h\dot I}{1 + h/\tau_e},
\qquad
\beta_{\mathrm{int}} = \frac{1}{1 + h/\tau_e}
$$

$K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$ 이면 mismatch 항이 0 이라
$I_{n+1}=\beta_{\mathrm{int}}I_n+(1-\beta_{\mathrm{int}})I_{\mathrm{des}}$ 이고,
두 값이 다르면 Method A 에서 다음 추가항이 붙는다.

$$
I_{n+1}=\beta_{\mathrm{int}}I_n+(1-\beta_{\mathrm{int}})I_{\mathrm{des}}
+ \frac{h}{1+h/\tau_e}\frac{(K_{e,\mathrm{nom}}g_r-K_{e,\mathrm{plant}}g_r)\omega}{L}
$$

기호:
- $I_n$, $I_{n+1}$: 현재 substep 시작/끝의 motor 전류
- $\dot I$: 전류 시간미분, 코드의 `act_dot`
- $I_{\mathrm{des}}$: PD 단계에서 계산되어 `ctrl` 로 전달되는 목표 전류
- $\tau_e$: 전기 시정수, $L/R$
- $L$: winding inductance
- $K_{e,\mathrm{nom}}g_r$: actuator 설정에 들어간 nominal back-EMF gain
- $K_{e,\mathrm{plant}}g_r$: plant 쪽 back-EMF gain
- $\omega$: actuator/joint velocity
- $\beta_{\mathrm{int}}$: Method A 전류 적분에 쓰는 implicit filter 계수
- $h$: 물리 적분 시간 간격

첫 번째 수식은 `_actuator_force` 안의
`act_dot = (ctrl - act) / tau_e` 와 mismatch 보정 `act_dot += ... * omega / L_dyn` 로
구현된다. 두 번째 수식은 `act = act_in + ... * opt_timestep / (1.0 + opt_timestep / tau)` 로 구현된다.

기계측 적분 (semi-implicit Euler: q̇ ← q̇ + h·q̈, q ← q + h·q̇_new) 본문은
`mujoco_warp/_src/forward.py:251-300` (`_advance`) — 호출은 `:352-379` (`euler`).
`_next_velocity` (`:114-128`) 가 `qvel ← qvel + h·qacc` 를, 이어서
`_next_position` (`:51-112`) 가 갱신된 `qvel` 로 `qpos ← qpos + h·qvel` 를 수행해
MuJoCo C 의 `mj_advance` 와 동일한 semi-implicit 순서를 따른다.


#### (B) 값 대입 — 1관절 적분

기계측 적분 본문은 `mujoco_warp/_src/forward.py:251-300` (`_advance`).
아래 셀의 `q̇ ← q̇ + h·q̈` → `q ← q + h·q̇_new` 순서는 `_next_velocity`
→ `_next_position` 호출 순서와 동일하다.

아래 코드는 기계측 semi-implicit Euler 와 전기측 적분을 각각 구현한다.

$$
\dot q_{n+1} = \dot q_n + h\ddot q_n,
\qquad
q_{n+1} = q_n + h\dot q_{n+1}
$$

$$
I_{n+1} = I_n + \frac{h}{1+h/\tau_e}\frac{I_{\mathrm{des}}-I_n}{\tau_e}
$$

기호:
- $q_n$, $q_{n+1}$: 현재/다음 substep 의 관절 위치
- $\dot q_n$, $\dot q_{n+1}$: 현재/다음 substep 의 관절속도
- $\ddot q_n$: 현재 substep 에서 dynamics 로 계산한 관절가속도
- $I_n$, $I_{n+1}$: 현재/다음 substep 의 motor 전류
- $I_{\mathrm{des}}$: PD 단계에서 전달된 목표 전류
- $h$: 물리 적분 시간 간격
- $\tau_e$: 전기 시정수

코드의 `q_dot_new`, `q_new`, `I_np1` 가 위 세 결과값이다.

In [ ]:
import numpy as np

# 4.2 에서 받은 q, q̇, q̈
q     = np.array([0.05])
q_dot = np.array([0.10])
q_ddot = np.array([qddot])    # 4.2 (B) 의 마지막 출력

dt = 1e-4    # = physics timestep   (이 값을 바꿔보세요)

# 기계측 적분 본문은 mujoco_warp/_src/forward.py:251-300 (_advance):
#   _next_velocity:114 (qvel += h*qacc) → _next_position:51 (qpos += h*qvel_new). 동일 순서.
# 기계측 semi-implicit Euler
q_dot_new = q_dot + dt * q_ddot
q_new     = q     + dt * q_dot_new
print(f"q̇_new = {q_dot_new}")
print(f"q_new  = {q_new}")

# 전기측: β_int (Method A) 전류 적분
Kt, Ke, gr = 0.128, 0.128, 6.33
R,  L      = 0.3, 1e-4
tau_e      = L / R

# act_dot 본문 (healthy: dynprm[3] = dynprm[1] → Ke mismatch 항 = 0)
I_n        = 1.0
ctrl_amps  = 5.0
act_dot    = (ctrl_amps - I_n) / tau_e

# Method A: I_{n+1} = I_n + h · act_dot / (1 + h/τ)
I_np1      = I_n + dt * act_dot / (1.0 + dt / tau_e)

# 등가 형식: I_{n+1} = β·I_n + (1−β)·ctrl,  β = 1/(1+h/τ)
beta_int   = 1.0 / (1.0 + dt / tau_e)
I_np1_eq   = beta_int * I_n + (1.0 - beta_int) * ctrl_amps

print(f"β_int      = {beta_int:.6f}")
print(f"I_{{n+1}}      = {I_np1:.6f} [A]")
print(f"I_{{n+1}} (β형) = {I_np1_eq:.6f} [A]   ← 두 형식이 일치해야 함")


#### (C) 중간 결과

한 substep (0.1 ms) 후의 상태 = `(q_new, q_dot_new, I_np1)`.
다음 substep 의 4.1 입력으로 들어간다.


### 4.4 한 substep 요약

화살표 한 줄:
$\tau_{\text{des}}, q, \dot q, I \to$ **운동방정식** $\to$
**β_imp (Schur · Force RHS)** $\to$ **β_int (적분)** $\to$
다음 $(q, \dot q, I)$.

아래 셀은 4.1 ∼ 4.3 의 수식

$$
M_{\mathrm{eff}} = M - hJ_c^\top sJ_c,
\qquad
F_{\mathrm{post}} = K_t g_r I + K_t g_r(1-\beta_{\mathrm{imp}})(I_{\mathrm{des}}-I),
\qquad
\ddot q = M_{\mathrm{eff}}^{-1}(F_{\mathrm{post}} - b\dot q - C_q - g_q),
\qquad
I_{n+1}=I_n+\frac{h\dot I}{1+h/\tau_e}
$$

기호:
- $M_{\mathrm{eff}}$: 한 substep dynamics 풀이에 쓰는 effective mass
- $M$: 원래 mass scalar 또는 mass matrix
- $J_c$: 이 통합 예제에서 1로 둔 contact/actuator Jacobian
- $s$: motor coupling Schur scalar, 기존 표기의 $-BD^{-1}C$
- $F_{\mathrm{post}}$: Schur RHS 보정 후 actuator torque
- $b\dot q$: viscous damping 토크
- $I$: 현재 전류, $I_{\mathrm{des}}$: 목표 전류
- $\beta_{\mathrm{imp}}$: Schur/Force RHS 쪽 implicit 계수
- $I_n$, $I_{n+1}$: 전류 적분 전/후 값
- $\dot I$: 전류 시간미분
- $h$: 물리 적분 시간 간격, $\tau_e$: 전기 시정수

을 `one_substep(...)` 함수로 묶어 1 PD 주기 (5 ms = 50 substep) 동안 반복하고,
(q, q̇, I) 변화를 표로 출력한다.


In [ ]:
import numpy as np
import pandas as pd

def one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params):
    # 1 자유도 축소 모형의 한 substep (4.1 → 4.2 → 4.3).
    Kt, Ke, gr = params['Kt'], params['Ke'], params['gr']
    R,  L,  h  = params['R'],  params['L'],  params['h']
    Kt_gr      = Kt * gr
    Ke_gr      = Ke * gr
    tau_e      = L / R

    # ── 4.2 β_imp ──────────────────────────────────────────
    omb        = h / (tau_e + h)              # 1 − β_imp
    schur      = -omb * Kt_gr * Ke_gr / R
    qderiv     = h * (1.0 * 1.0 * schur)      # J=1
    M_eff      = M - qderiv                    # M_eff = M − h·qDeriv

    gain       = Kt_gr
    force      = gain * I + gain * omb * (ctrl - I)   # Schur RHS 보정 후 actuator torque

    # ── 4.1 비구속 가속도 (접촉 무시) ───────────────────────
    damping_force = damping_b * q_dot
    qddot      = (force - damping_force - C_q - g_q_v) / M_eff

    # ── 4.3 적분 ────────────────────────────────────────────
    q_dot_new  = q_dot + h * qddot
    q_new      = q     + h * q_dot_new

    act_dot    = (ctrl - I) / tau_e            # healthy
    I_new      = I + h * act_dot / (1.0 + h / tau_e)

    return q_new, q_dot_new, I_new


# 파라미터 (이 값을 바꿔보세요)
params = dict(Kt=0.128, Ke=0.128, gr=6.33, R=0.3, L=1e-4, h=1e-4)

ctrl = 5.0      # = I_des [A]

# 단순화한 1자유도 동역학 (이 값을 바꿔보세요)
M, damping_b, C_q, g_q_v = 0.5, 0.30, 0.0, 0.0

# 초기 상태
q, q_dot, I = 0.0, 0.0, 0.0

N_SUBSTEPS_PER_PD = 50    # 5 ms / 0.1 ms (이 값을 바꿔보세요)

rows = [{"k": 0, "q": q, "q_dot": q_dot, "I": I}]
for k in range(1, N_SUBSTEPS_PER_PD + 1):
    q, q_dot, I = one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params)
    rows.append({"k": k, "q": q, "q_dot": q_dot, "I": I})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda x: f"{x: .6f}"))


## 5. 전체 흐름도

```mermaid
flowchart TB
    subgraph POL["정책 (20 ms)"]
        OBS["관측: v_body, ω_body, g_proj, v_cmd, q, q̇, q_des_prev"]
        OBS -- "신경망" --> QDES["q_des"]
    end
    subgraph PDC["PD (5 ms × 4)"]
        QDES --> PDF["τ_des = clip( Kp·(q_des − q) − Kd·q̇,  ±τ_max(ω) )"]
        PDF  --> IDS["I_des = τ_des / (Kt·gr)"]
    end
    subgraph PHY["물리 (0.1 ms × 50 per PD)"]
        IDS  --> EOM["M(q)·q̈ + b·q̇ + C(q,q̇) + g(q) = Kt·gr·I + J_cᵀλ"]
        EOM  --> SCH["β_imp · Schur:  qDeriv += −(1−β_imp)·Kt·gr·Ke·gr/R · J_cᵀJ_c<br/>force += gain·(1−β_imp)·(ctrl − I_old)"]
        SCH  --> INT["β_int · 적분:  I_{n+1} = I_n + h·act_dot / (1 + h/τ)<br/>q̇_{n+1} = q̇_n + h·q̈,  q_{n+1} = q_n + h·q̇_{n+1}"]
        INT  --> EOM
    end
    PHY -- "(q, q̇, I) 마지막 substep" --> POL
```

기호:
- $q_{\mathrm{des}}$: 정책이 출력한 목표 관절각
- $\tau_{\mathrm{des}}$: PD와 motor saturation 후 목표 토크
- $I_{\mathrm{des}}$: 목표 전류
- $M(q)$, $C(q,\dot q)$, $g(q)$: mass matrix, 속도 의존 일반화 힘, 중력 일반화 힘
- $J_c^\top\lambda$: 접촉 constraint force 를 관절공간으로 투영한 항
- $\beta_{\mathrm{imp}}$: Schur/Force RHS 보정 계수
- $\beta_{\mathrm{int}}$: 전류 적분 계수
- $h$: 물리 적분 시간 간격


## 6. 한 정책 주기 누적 호출 횟수

| 단계 | 주기 | 정책 한 주기당 호출 횟수 |
|------|------|--------------------------|
| 정책 | 20 ms | 1 |
| PD | 5 ms | 4 (= 20 ms / 5 ms) |
| 물리 | 0.1 ms | 200 (= 20 ms / 0.1 ms = `decimation`) |

PD 1 회당 물리 substep 수 = 50 (= 5 ms / 0.1 ms = `pd_substeps`).


## 7. 구현 위치 참조표

"본문 기호" 컬럼은 노트의 식·기호와 코드의 대응을 보여준다. 본 표는
학습/플레이 시 실제로 호출되는 mjlab + mujoco_warp Python 경로만 다룬다
(C 엔진은 GPU 학습 경로에서 호출되지 않으므로 생략).

기호:
- $h$ : 물리 적분 시간 간격 (= `m->opt.timestep`)
- $M$ : mass matrix (`d->qM`, factor 후 `d->qLD`)
- $C(q,\dot q),\,g(q)$ : Coriolis·중력 일반화 힘. mujoco 에서는 이 둘이
  `qfrc_bias` 한 벡터로 합쳐서 RNE 로 계산된다.
- $J_c$ : 접촉/등식·부등식 constraint Jacobian (`d->efc_J`,
  행 = `nefc`, 열 = `nv`)
- $\lambda$ : constraint force/impulse (`d->efc_force`, 관절공간 투영은 `d->qfrc_constraint`)
- $\ddot q_{\mathrm{free}}$ : 비구속 가속도 (`d->qacc_smooth`)
  $= M^{-1}(\,$`qfrc_passive` $-$ `qfrc_bias` $+$ `qfrc_applied` $+$ `qfrc_actuator`$)$

| 단계 | 본문 기호 | 코드상 명칭 | 파일 | 라인 |
|------|----------|-----------|------|------|
| 정책 | task 등록 | `register_mjlab_task("Unitree-Go2-Flat-MethodA-Electric", ...)` | `src/tasks/velocity/config/go2/__init__.py` | 81-93 |
| 정책 | 시간 설정 | `cfg.sim.mujoco.timestep`, `cfg.decimation` | `src/tasks/velocity/config/go2/env_cfgs.py` | 206-207 |
| PD | substep 상수 | `_COUPLED_SUBSTEPS = 200`, `_PD_RECOMPUTE = 50` | `src/assets/robots/unitree_go2/go2_constants.py` | 305-306 |
| PD | actuator cfg | `_MA_MOTOR`, `GO2_METHODA_HIP/THIGH/CALF` | `src/assets/robots/unitree_go2/go2_constants.py` | 342-358 |
| PD | $\tau_{\mathrm{des}}$ | `tau_des = super().compute(cmd)` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 509 |
| PD | $I_{\mathrm{des}}$ | `I_des = tau_des / self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 531 |
| PD | $I_{\mathrm{des}}(t)=I_{\mathrm{des}}(t_m)$ 값 유지 | `_I_des_hold`, `pd_substeps` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 497-540 |
| 물리 | dynprm[4] = method | `act.dynprm[4] = _METHOD_TO_DYNPRM4[cfg.method]` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 333 |
| 물리 | $1-\beta_{\mathrm{imp}}$ (Schur, A 분기) | `one_minus_beta = h_dt / (tau_e + h_dt)` | `mujoco_warp/_src/derivative.py` | 87 |
| 물리 | Schur scalar $s$ | `schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` | `mujoco_warp/_src/derivative.py` | 89 |
| 물리 | qDeriv 누적 ($J_c^\top s J_c$) | `qderiv_contrib += moment_i * moment_j * vel` | `mujoco_warp/_src/derivative.py` | 170 |
| 물리 | $M_{\mathrm{eff}} = qM - h \cdot$ qDeriv | `qM_in - qderiv` (after `qderiv *= h`) | `mujoco_warp/_src/derivative.py` | 209-217 |
| 물리 | $\beta_{\mathrm{imp}}$ Force RHS | `force += gain * one_minus_beta * (ctrl - I_old)` | `mujoco_warp/_src/forward.py` | 765-787 |
| 물리 | $\dot I$ (filterexact) | `act_dot = (ctrl - act) / tau_e + (dynprm[3]-dynprm[1])*omega/L` | `mujoco_warp/_src/forward.py` | 692-704 |
| 물리 | $\beta_{\mathrm{int}}$ (Method A) | `act = act_in + ... * h / (1 + h/τ)` | `mujoco_warp/_src/forward.py` | 163-164 |
| 물리 | force = $K_t g_r I$ | `act.gainprm[0] = self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 351-353 |
| PD | PD 수식 본문 ($\tau_{pd}=K_p\,e_q-K_d\,\dot q$) | `IdealPdActuator.compute` | `mjlab/actuator/pd_actuator.py` | 96-107 |
| PD | DC 모터 saturation (torque-speed clip) | `DcMotorActuator._clip_effort` | `mjlab/actuator/dc_actuator.py` | 136-162 |
| 물리 (mjwarp) | 접촉 efc 행 어셈블리 (pyramidal) | `_contact_pyramidal` | `mujoco_warp/_src/constraint.py` | 1536-1768 |
| 물리 (mjwarp) | 접촉 efc 행 어셈블리 (elliptic) | `_contact_elliptic` | `mujoco_warp/_src/constraint.py` | 1770-2000 |
| 물리 (mjwarp) | constraint 어셈블리 (디스패처) | `make_constraint` | `mujoco_warp/_src/constraint.py` | 2002 |
| 물리 (mjwarp) | $\lambda$ 풀이 본문 (PCG/Newton) | `solver.solve` → `_solve` → `_solver_iteration` | `mujoco_warp/_src/solver.py` | 3350 / 3359 / 3241 |
| 물리 (mjwarp) | qfrc_constraint = $J_c^\top\lambda$ | `_update_constraint` 안 `update_constraint_init_qfrc_constraint_*` | `mujoco_warp/_src/solver.py` | 2154-2202 |
| 물리 (mjwarp) | semi-implicit Euler ($\dot q\mathrel{+}=h\ddot q$, $q\mathrel{+}=h\dot q$) | `_advance` (`_next_velocity` → `_next_position`) | `mujoco_warp/_src/forward.py` | 251-300 (114, 51) |
| 물리 (mjwarp) | Euler integrator 디스패치 (damping 분기 포함) | `euler` | `mujoco_warp/_src/forward.py` | 352-379 |
